In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os
import re
import string
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. PATH CONFIGURATION
base_path = r"C:\Users\prash\OneDrive\Desktop\Spam detection"
zip_path = os.path.join(base_path, "archive (6).zip")
output_dir = os.path.join(base_path, "Model_Results")

# Create output folder automatically
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. DATA LOADING & CLEANING
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

# Find the CSV file inside the zip
files = os.listdir(output_dir)
csv_file = [f for f in files if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(output_dir, csv_file), encoding='latin-1')

# Clean columns (Handles standard Kaggle v1, v2 format)
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'message']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

def clean_text(text):
    text = str(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    return re.sub(r'\d+', '', text)

df['message'] = df['message'].apply(clean_text)

# 3. VECTORIZATION
tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
X = tfidf.fit_transform(df['message'])
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TRAIN, EVALUATE, AND SAVE EVERYTHING
models = {
    "Naive_Bayes": MultinomialNB(),
    "Logistic_Regression": LogisticRegression(),
    "SVM": SVC(kernel='linear', probability=True)
}

performance = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # Save Model File
    joblib.dump(model, os.path.join(output_dir, f"{name}_model.pkl"))
    
    # Calculate Accuracy
    acc = accuracy_score(y_test, y_pred)
    performance.append({'Algorithm': name, 'Accuracy': acc})
    
    # --- CONFUSION MATRIX CALCULATION & SAVING ---
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Ham', 'Spam'], yticklabels=['Ham', 'Spam'])
    plt.title(f'Confusion Matrix - {name}\nAccuracy: {acc:.4f}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    
    # Save the plot as an image
    plt.savefig(os.path.join(output_dir, f"Confusion_Matrix_{name}.png"))
    plt.close() # Closes the plot so it doesn't block the script

# 5. SAVE COMPARISON CHART
perf_df = pd.DataFrame(performance)
plt.figure(figsize=(8, 5))
sns.barplot(x='Algorithm', y='Accuracy', data=perf_df, palette='viridis')
plt.title('Algorithm Comparison')
plt.ylim(0.9, 1.0)
plt.savefig(os.path.join(output_dir, "Overall_Comparison.png"))
plt.close()

# Save Vectorizer (Required for future predictions)
joblib.dump(tfidf, os.path.join(output_dir, "vectorizer.pkl"))

print(f"\n--- SUCCESS ---")
print(f"All files are saved here: {output_dir}")
print(perf_df)


Training Naive_Bayes...
Training Logistic_Regression...
Training SVM...

--- SUCCESS ---
All files are saved here: C:\Users\prash\OneDrive\Desktop\Spam detection\Model_Results
             Algorithm  Accuracy
0          Naive_Bayes  0.973094
1  Logistic_Regression  0.955157
2                  SVM  0.981166


C:\Users\prash\AppData\Local\Temp\ipykernel_32896\2598697366.py:89: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Algorithm', y='Accuracy', data=perf_df, palette='viridis')
